# Comparison of the 500k models: M00 vs M07

This notebook compares a trained `SimpleCNN_Baseline` and `SimpleCNN_MultiHead` models using the same 500k dataset:

- **M00_bs256_batchslices**: baseline trained with bs256 and `HDF5BatchIterableDataset`.
- **M07_MultiHead**: includes a independent head for each label.

The comparison is made on the common **validation, calibration and test samples**. It includes:

1. file and split integrity checks;
2. standardized and physical-space metrics;
3. absolute-error quantiles;
4. regression-to-the-mean diagnostics for all three labels;
5. bias and error as a function of the true parameter;
6. direct deltas between the two models;
7. training-history comparison.

The final decision should be based primarily on the **test split**. Validation is useful for model development, while calibration should remain reserved for conformal calibration.


In [ ]:
from pathlib import Path
import os
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run the notebook either from the repository root or from its notebooks/ directory.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name != "cbc_pe" and (PROJECT_ROOT / "cbc_pe").exists():
    PROJECT_ROOT = PROJECT_ROOT / "cbc_pe"

# Local synchronized artifacts. raw/ and processed/ are intentionally absent locally.
DATA_ROOT = Path("/data/vserrano/cbc_pe_data")
RESULTS_DIR = DATA_ROOT / "results"
CHECKPOINTS_DIR = DATA_ROOT / "models" / "checkpoints"

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "outputs" / "comparison_500k_M00_vs_M07"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_NAMES_DEFAULT = ["chirp_mass", "total_mass", "chi_eff"]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("RESULTS_DIR exists:", RESULTS_DIR.exists())
print("CHECKPOINTS_DIR exists:", CHECKPOINTS_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)


## 1. Register the two runs

In [ ]:
DATASET_ID = "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000"

RUNS = {
    "M00": {
        "prediction": RESULTS_DIR / DATASET_ID / (
            "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000_"
            "SimpleCNN_Baseline_batchslices_bs256_MSELoss_seed123_"
            "val_cal_test_predictions_embeddings.npz"
        ),
        "history": RESULTS_DIR / DATASET_ID / (
            "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000_"
            "SimpleCNN_Baseline_batchslices_bs256_MSELoss_seed123_history.npz"
        ),
        "checkpoint": CHECKPOINTS_DIR / DATASET_ID / (
            "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000_"
            "SimpleCNN_Baseline_batchslices_bs256_MSELoss_seed123_checkpoint.pt"
        ),
    },
    "M07": {
        "prediction": RESULTS_DIR / DATASET_ID / (
            "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000_"
            "SimpleCNN_MultiHead_M07_multihead_emb64_head32_MSELoss_seed123_"
            "val_cal_test_predictions_embeddings.npz"
        ),
        "history": RESULTS_DIR / DATASET_ID / (
            "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000_"
            "SimpleCNN_MultiHead_M07_multihead_emb64_head32_MSELoss_seed123_history.npz"
        ),
        "checkpoint": CHECKPOINTS_DIR / DATASET_ID / (
            "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000_"
            "SimpleCNN_MultiHead_M07_multihead_emb64_head32_MSELoss_seed123_checkpoint.pt"
        ),
    },
}

file_rows = []
for run_id, paths in RUNS.items():
    row = {"run_id": run_id}
    for kind, path in paths.items():
        row[f"{kind}_exists"] = path.exists()
        row[f"{kind}_path"] = str(path)
        row[f"{kind}_size_MB"] = path.stat().st_size / 1024**2 if path.exists() else np.nan
    file_rows.append(row)

files_df = pd.DataFrame(file_rows)
files_df


In [ ]:
missing_predictions = [
    run_id for run_id, paths in RUNS.items()
    if not paths["prediction"].exists()
]
assert not missing_predictions, f"Missing prediction files for: {missing_predictions}"


## 2. Load files and verify compatibility

In [ ]:
def load_prediction_file(path):
    with np.load(path, allow_pickle=True) as data:
        return {key: data[key] for key in data.files}


run_data = {
    run_id: load_prediction_file(paths["prediction"])
    for run_id, paths in RUNS.items()
}

for run_id, data in run_data.items():
    available_splits = [
        split for split in ("train", "val", "cal", "test")
        if f"pred_{split}" in data and f"y_{split}" in data
    ]
    print("\n", run_id)
    print("  splits:", available_splits)
    print("  labels:", [str(x) for x in data.get("label_names", LABEL_NAMES_DEFAULT).tolist()])
    for split in available_splits:
        print(
            f"  {split:5s}",
            "pred", data[f"pred_{split}"].shape,
            "emb", data[f"emb_{split}"].shape,
            "y", data[f"y_{split}"].shape,
            "idx", data[f"idx_{split}"].shape,
        )


In [ ]:
reference_id = "M00"
reference = run_data[reference_id]

label_names = [
    str(x) for x in reference.get("label_names", np.array(LABEL_NAMES_DEFAULT)).tolist()
]
y_mean = np.asarray(reference["y_mean"], dtype=np.float64)
y_std = np.asarray(reference["y_std"], dtype=np.float64)

for run_id, data in run_data.items():
    np.testing.assert_allclose(data["y_mean"], y_mean, rtol=0, atol=1e-7)
    np.testing.assert_allclose(data["y_std"], y_std, rtol=0, atol=1e-7)

print("Compatible label statistics.")
print("label_names:", label_names)
print("y_mean:", y_mean)
print("y_std:", y_std)


### Align samples by physical HDF5 index

The two prediction files may store each split in a different row order. The comparison therefore aligns them using `idx_val`, `idx_cal` and `idx_test` rather than assuming that row `i` refers to the same event.


In [ ]:
def aligned_split(run_data, split, reference_run="M00"):
    common = None

    for data in run_data.values():
        idx = np.asarray(data[f"idx_{split}"], dtype=np.int64)
        common = idx if common is None else np.intersect1d(common, idx)

    common = np.asarray(common, dtype=np.int64)
    aligned = {}

    for run_id, data in run_data.items():
        idx = np.asarray(data[f"idx_{split}"], dtype=np.int64)
        order = np.argsort(idx)
        idx_sorted = idx[order]

        positions = np.searchsorted(idx_sorted, common)
        if not np.array_equal(idx_sorted[positions], common):
            raise ValueError(f"Could not align all {split} indices for {run_id}")

        aligned[run_id] = {
            "idx": common,
            "pred": np.asarray(data[f"pred_{split}"])[order][positions],
            "y": np.asarray(data[f"y_{split}"])[order][positions],
            "emb": np.asarray(data[f"emb_{split}"])[order][positions],
        }

    y_ref = aligned[reference_run]["y"]
    for run_id, arrays in aligned.items():
        np.testing.assert_allclose(
            arrays["y"], y_ref,
            rtol=1e-5,
            atol=1e-5,
            err_msg=f"Target mismatch after aligning {split}: {run_id}",
        )

    return aligned


ALIGNED = {
    split: aligned_split(run_data, split)
    for split in ("val", "cal", "test")
}

for split, models in ALIGNED.items():
    print(split, "common samples:", len(next(iter(models.values()))["idx"]))


## 3. Metric helpers

In [ ]:
def regression_metrics(y_true, y_pred):
    # Sign convention used throughout this notebook:
    # residual = prediction - truth
    residual = y_pred - y_true
    abs_error = np.abs(residual)

    mse = np.mean(residual**2, axis=0)
    rmse = np.sqrt(mse)
    mae = np.mean(abs_error, axis=0)
    bias = np.mean(residual, axis=0)
    median_abs_error = np.median(abs_error, axis=0)
    residual_std = np.std(residual, axis=0)

    ss_res = np.sum(residual**2, axis=0)
    ss_tot = np.sum((y_true - np.mean(y_true, axis=0))**2, axis=0)
    r2 = 1.0 - ss_res / ss_tot

    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "bias": bias,
        "median_abs_error": median_abs_error,
        "residual_std": residual_std,
        "r2": r2,
        "global_mse": float(np.mean(residual**2)),
        "global_rmse": float(np.sqrt(np.mean(residual**2))),
        "global_mae": float(np.mean(abs_error)),
    }


def to_physical(y_standardized):
    return y_standardized * y_std + y_mean


def slope_diagnostics(y_true, y_pred):
    rows = []
    for j, label in enumerate(label_names):
        slope, intercept = np.polyfit(y_true[:, j], y_pred[:, j], deg=1)
        correlation = np.corrcoef(y_true[:, j], y_pred[:, j])[0, 1]
        rows.append({
            "label": label,
            "slope": float(slope),
            "intercept": float(intercept),
            "correlation": float(correlation),
            "true_min": float(y_true[:, j].min()),
            "true_max": float(y_true[:, j].max()),
            "pred_min": float(y_pred[:, j].min()),
            "pred_max": float(y_pred[:, j].max()),
            "range_ratio_pred_over_true": float(
                np.ptp(y_pred[:, j]) / np.ptp(y_true[:, j])
            ),
            "std_ratio_pred_over_true": float(
                np.std(y_pred[:, j]) / np.std(y_true[:, j])
            ),
            "q01_q99_ratio_pred_over_true": float(
                (
                    np.quantile(y_pred[:, j], 0.99)
                    - np.quantile(y_pred[:, j], 0.01)
                )
                /
                (
                    np.quantile(y_true[:, j], 0.99)
                    - np.quantile(y_true[:, j], 0.01)
                )
            ),
        })
    return rows


## 4. Compute metrics on val/cal/test

In [ ]:
metric_rows = []
quantile_rows = []
slope_rows = []

for split, models in ALIGNED.items():
    for run_id, arrays in models.items():
        for space in ("standardized", "physical"):
            if space == "standardized":
                y_true = arrays["y"]
                y_pred = arrays["pred"]
            else:
                y_true = to_physical(arrays["y"])
                y_pred = to_physical(arrays["pred"])

            metrics = regression_metrics(y_true, y_pred)

            metric_rows.append({
                "run_id": run_id,
                "split": split,
                "space": space,
                "label": "global",
                "MSE": metrics["global_mse"],
                "RMSE": metrics["global_rmse"],
                "MAE": metrics["global_mae"],
                "Bias_pred_minus_true": np.nan,
                "Median_abs_error": np.nan,
                "Residual_std": np.nan,
                "R2": np.nan,
                "n_samples": len(y_true),
            })

            abs_error = np.abs(y_pred - y_true)

            for j, label in enumerate(label_names):
                metric_rows.append({
                    "run_id": run_id,
                    "split": split,
                    "space": space,
                    "label": label,
                    "MSE": metrics["mse"][j],
                    "RMSE": metrics["rmse"][j],
                    "MAE": metrics["mae"][j],
                    "Bias_pred_minus_true": metrics["bias"][j],
                    "Median_abs_error": metrics["median_abs_error"][j],
                    "Residual_std": metrics["residual_std"][j],
                    "R2": metrics["r2"][j],
                    "n_samples": len(y_true),
                })

                q50, q68, q90, q95, q99 = np.quantile(
                    abs_error[:, j], [0.50, 0.68, 0.90, 0.95, 0.99]
                )
                quantile_rows.append({
                    "run_id": run_id,
                    "split": split,
                    "space": space,
                    "label": label,
                    "q50": q50,
                    "q68": q68,
                    "q90": q90,
                    "q95": q95,
                    "q99": q99,
                })

            if space == "physical":
                for row in slope_diagnostics(y_true, y_pred):
                    row.update({"run_id": run_id, "split": split})
                    slope_rows.append(row)

metrics_df = pd.DataFrame(metric_rows)
quantiles_df = pd.DataFrame(quantile_rows)
slopes_df = pd.DataFrame(slope_rows)


### Main test results: standardized space

In [ ]:
test_std = metrics_df.query(
    "split == 'test' and space == 'standardized'"
).copy()

test_std.sort_values(["label", "MSE"])


### Main test results: physical space

In [ ]:
test_phys = metrics_df.query(
    "split == 'test' and space == 'physical' and label != 'global'"
).copy()

test_phys.sort_values(["label", "RMSE"])


## 5. Direct deltas: M07 relative to M00

In [ ]:
def paired_delta_table(df, value_columns, index_columns):
    wide = df.pivot(index=index_columns, columns="run_id", values=value_columns)

    rows = []
    for index_values, row in wide.iterrows():
        if not isinstance(index_values, tuple):
            index_values = (index_values,)

        out = dict(zip(index_columns, index_values))
        for metric in value_columns:
            old = row[(metric, "M00")]
            new = row[(metric, "M07")]
            out[f"{metric}_M00"] = old
            out[f"{metric}_M07"] = new
            out[f"{metric}_delta"] = new - old
            out[f"{metric}_relative_change_pct"] = 100.0 * (new - old) / old
        rows.append(out)

    return pd.DataFrame(rows)


test_delta_df = paired_delta_table(
    metrics_df.query(
        "split == 'test' and space == 'standardized'"
    ),
    value_columns=["MSE", "MAE", "R2"],
    index_columns=["label"],
)

test_delta_df


Interpretation:

- Negative MSE or MAE relative change means M07 improved over M00.
- Positive R² delta means M07 improved over M00.
- Small changes should not be overinterpreted without checking whether the pattern is consistent across val, cal and test.


In [ ]:
global_stability_df = metrics_df.query(
    "space == 'standardized' and label == 'global'"
).pivot(index="split", columns="run_id", values="MSE")

global_stability_df["delta_M07_minus_M00"] = (
    global_stability_df["M07"]
    - global_stability_df["M00"]
)
global_stability_df["relative_change_pct"] = (
    100.0
    * global_stability_df["delta_M07_minus_M00"]
    / global_stability_df["M00"]
)

global_stability_df


### 5.1 Event-wise mean comparison

In [ ]:
def paired_bootstrap_delta(
    y_true,
    pred_a,
    pred_b,
    metric="squared_error",
    n_boot=2000,
    seed=123,
):
    rng = np.random.default_rng(seed)
    n = len(y_true)

    if metric == "squared_error":
        err_a = (pred_a - y_true) ** 2
        err_b = (pred_b - y_true) ** 2
    elif metric == "absolute_error":
        err_a = np.abs(pred_a - y_true)
        err_b = np.abs(pred_b - y_true)
    else:
        raise ValueError(metric)

    delta = err_b - err_a
    observed = delta.mean(axis=0)

    boot = np.empty((n_boot, y_true.shape[1]))

    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot[b] = delta[idx].mean(axis=0)

    lower = np.quantile(boot, 0.025, axis=0)
    upper = np.quantile(boot, 0.975, axis=0)

    return observed, lower, upper

In [ ]:
test = ALIGNED["test"]

y_true = test["M00"]["y"]
pred_m00 = test["M00"]["pred"]
pred_m07 = test["M07"]["pred"]

mean_delta, ci_low, ci_high = paired_bootstrap_delta(
    y_true,
    pred_m00,
    pred_m07,
    metric="squared_error",
)

In [ ]:
bootstrap_df = pd.DataFrame({
    "label": label_names,
    "mean_delta_M07_minus_M00": mean_delta,
    "ci95_low": ci_low,
    "ci95_high": ci_high,
    "supports_improvement": ci_high < 0,
})

bootstrap_df

Interpretación:

- intervalo completamente por debajo de 0: evidencia de mejora de M07;
- intervalo cruza 0: diferencia compatible con ruido estadístico;
- intervalo completamente por encima de 0: M07 empeora.

Con 30.000 muestras es posible que diferencias pequeñas sean estadísticamente detectables, pero eso no implica que sean científicamente relevantes. Por eso conservaría también el porcentaje relativ

In [ ]:
mae_delta, mae_ci_low, mae_ci_high = paired_bootstrap_delta(
    y_true,
    pred_m00,
    pred_m07,
    metric="absolute_error",
)

mae_bootstrap_df = pd.DataFrame({
    "label": label_names,
    "mean_delta_M07_minus_M00": mae_delta,
    "ci95_low": mae_ci_low,
    "ci95_high": mae_ci_high,
    "supports_improvement": mae_ci_high < 0,
})

mae_bootstrap_df

Incluimos un "win-rate" por evento

In [ ]:
def per_sample_win_rate(y_true, pred_m00, pred_m07):
    ae_m00 = np.abs(pred_m00 - y_true)
    ae_m07 = np.abs(pred_m07 - y_true)

    return pd.DataFrame({
        "label": label_names,
        "M07_win_rate": np.mean(ae_m07 < ae_m00, axis=0),
        "tie_rate": np.mean(ae_m07 == ae_m00, axis=0),
        "M00_win_rate": np.mean(ae_m00 < ae_m07, axis=0),
    })

In [ ]:
win_rate_df = per_sample_win_rate(
    y_true,
    pred_m00,
    pred_m07,
)

win_rate_df

## 6. Regression-to-the-mean diagnostics

In [ ]:
slopes_df.query("split == 'test'").sort_values(["label", "run_id"])


For the fit

$$
\widehat{y} = a\,y + b,
$$

a slope `a < 1` indicates contraction toward the centre of the target distribution. The closer the slope and range ratio are to 1, the less severe the contraction.


In [ ]:
test_slope_delta = slopes_df.query("split == 'test'").pivot(
    index="label",
    columns="run_id",
    values=["slope", "range_ratio_pred_over_true", "correlation"],
)

test_slope_delta


### Predicted versus true values

In [ ]:
rng = np.random.default_rng(123)
split = "test"
max_points = 7000

for j, label in enumerate(label_names):
    plt.figure(figsize=(6.5, 5.5))

    for run_id, arrays in ALIGNED[split].items():
        y_true = to_physical(arrays["y"])[:, j]
        y_pred = to_physical(arrays["pred"])[:, j]

        n_plot = min(max_points, len(y_true))
        positions = rng.choice(len(y_true), size=n_plot, replace=False)

        plt.scatter(
            y_true[positions],
            y_pred[positions],
            s=7,
            alpha=0.5,
            label=run_id,
        )

    all_true = np.concatenate([
        to_physical(arrays["y"])[:, j]
        for arrays in ALIGNED[split].values()
    ])
    lo, hi = all_true.min(), all_true.max()
    plt.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1.2, label="ideal")

    plt.xlabel(f"True {label}")
    plt.ylabel(f"Predicted {label}")
    plt.title(f"Test: predicted vs true — {label}")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 7. Bias and error across the true-parameter range

In [ ]:
def binned_true_diagnostics(y_true, y_pred, n_bins=8):
    rows = []

    for j, label in enumerate(label_names):
        # Shared quantile edges ensure enough samples in every bin.
        edges = np.quantile(y_true[:, j], np.linspace(0, 1, n_bins + 1))
        edges = np.unique(edges)

        bin_idx = np.digitize(y_true[:, j], edges[1:-1], right=False)

        for b in range(len(edges) - 1):
            mask = bin_idx == b
            if not np.any(mask):
                continue

            residual = y_pred[mask, j] - y_true[mask, j]
            abs_error = np.abs(residual)

            rows.append({
                "label": label,
                "bin": b,
                "low": edges[b],
                "high": edges[b + 1],
                "true_mean": y_true[mask, j].mean(),
                "pred_mean": y_pred[mask, j].mean(),
                "n_samples": int(mask.sum()),
                "bias_pred_minus_true": residual.mean(),
                "MAE": abs_error.mean(),
                "RMSE": np.sqrt(np.mean(residual**2)),
                "q90_abs_error": np.quantile(abs_error, 0.90),
            })

    return pd.DataFrame(rows)


binned_rows = []
for run_id, arrays in ALIGNED["test"].items():
    y_true_phys = to_physical(arrays["y"])
    y_pred_phys = to_physical(arrays["pred"])

    diag = binned_true_diagnostics(y_true_phys, y_pred_phys, n_bins=8)
    diag.insert(0, "run_id", run_id)
    binned_rows.append(diag)

binned_df = pd.concat(binned_rows, ignore_index=True)
binned_df.head()


In [ ]:
for label in label_names:
    subset = binned_df.query("label == @label")

    plt.figure(figsize=(7.5, 4.5))
    for run_id, group in subset.groupby("run_id"):
        plt.plot(
            group["true_mean"],
            group["bias_pred_minus_true"],
            marker="o",
            label=run_id,
        )

    plt.axhline(0.0, linestyle="--", linewidth=1)
    plt.xlabel(f"Mean true {label} in bin")
    plt.ylabel("Mean bias: prediction − truth")
    plt.title(f"Test bias by true-value bin — {label}")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
for label in label_names:
    subset = binned_df.query("label == @label")

    plt.figure(figsize=(7.5, 4.5))
    for run_id, group in subset.groupby("run_id"):
        plt.plot(
            group["true_mean"],
            group["MAE"],
            marker="o",
            label=run_id,
        )

    plt.xlabel(f"Mean true {label} in bin")
    plt.ylabel("MAE")
    plt.title(f"Test MAE by true-value bin — {label}")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 8. Training-history comparison

In [ ]:
def load_history(path):
    with np.load(path, allow_pickle=True) as data:
        if "train_loss" in data.files and "val_loss" in data.files:
            train_loss = np.asarray(data["train_loss"], dtype=float)
            val_loss = np.asarray(data["val_loss"], dtype=float)
        elif "history" in data.files:
            obj = data["history"].tolist()
            if not isinstance(obj, dict):
                raise ValueError(f"Unsupported history format: {path}")
            train_loss = np.asarray(obj["train_loss"], dtype=float)
            val_loss = np.asarray(obj["val_loss"], dtype=float)
        else:
            raise KeyError(f"No train_loss/val_loss in {path}")

    return pd.DataFrame({
        "epoch": np.arange(1, len(train_loss) + 1),
        "train_loss": train_loss,
        "val_loss": val_loss,
    })


history_frames = {}
history_rows = []

for run_id, paths in RUNS.items():
    if not paths["history"].exists():
        print("Missing history:", run_id, paths["history"])
        continue

    history = load_history(paths["history"])
    history_frames[run_id] = history

    best_position = history["val_loss"].idxmin()
    history_rows.append({
        "run_id": run_id,
        "best_epoch": int(history.loc[best_position, "epoch"]),
        "stop_epoch": int(history["epoch"].iloc[-1]),
        "best_val_loss": float(history.loc[best_position, "val_loss"]),
        "train_loss_at_best": float(history.loc[best_position, "train_loss"]),
        "generalization_gap_at_best": float(
            history.loc[best_position, "val_loss"]
            - history.loc[best_position, "train_loss"]
        ),
    })

history_summary_df = pd.DataFrame(history_rows)
history_summary_df


In [ ]:
plt.figure(figsize=(9, 5.5))

for run_id, history in history_frames.items():
    plt.plot(
        history["epoch"],
        history["val_loss"],
        label=f"{run_id} val",
    )

plt.xlabel("Epoch")
plt.ylabel("Standardized validation MSE")
plt.title("Validation histories")
plt.legend()
plt.tight_layout()
plt.show()


### Important caveat about epoch comparisons

An epoch is not equivalent across batch sizes:

- bs64 performs roughly four times as many optimizer updates per epoch as bs256;
- compare both **epochs** and approximate optimizer steps;
- wall-clock time should be read from the checkpoint metadata or training logs.


In [ ]:
TRAIN_SIZE = 400_000
BATCH_SIZE_BY_RUN = {
    "M00": 256,
    "M07": 256,
}

if not history_summary_df.empty:
    history_summary_df["updates_per_epoch_approx"] = history_summary_df["run_id"].map(
        lambda run_id: TRAIN_SIZE // BATCH_SIZE_BY_RUN[run_id]
    )
    history_summary_df["optimizer_steps_at_best_approx"] = (
        history_summary_df["best_epoch"]
        * history_summary_df["updates_per_epoch_approx"]
    )

history_summary_df


## 9. Save comparison tables

In [ ]:
metrics_df.to_csv(OUTPUT_DIR / "metrics_all_splits.csv", index=False)
quantiles_df.to_csv(OUTPUT_DIR / "absolute_error_quantiles.csv", index=False)
slopes_df.to_csv(OUTPUT_DIR / "slope_diagnostics.csv", index=False)
binned_df.to_csv(OUTPUT_DIR / "binned_true_diagnostics_test.csv", index=False)
test_delta_df.to_csv(OUTPUT_DIR / "test_deltas_M00_vs_M07.csv", index=False)
global_stability_df.to_csv(OUTPUT_DIR / "global_mse_stability.csv")
history_summary_df.to_csv(OUTPUT_DIR / "training_history_summary.csv", index=False)

print("Saved comparison tables to:", OUTPUT_DIR)


## 10. Decision

M07 improves the standardized global test MSE by approximately 1.7% and
the chi_eff MSE by approximately 2.4%. However, the chi_eff regression
slope changes only from 0.782 to 0.784, indicating that the multi-head
architecture does not materially reduce regression-to-the-mean.

Conclusion:
- M07 provides a small error reduction.
- The improvement is not large enough to explain the main bottleneck.
- The shared encoder and limited temporal receptive field remain the
  leading hypotheses.
- M00 remains the operational baseline unless paired uncertainty analysis
  confirms a robust and practically relevant advantage for M07.

___________________

# 10*. Deltas per bin

In [ ]:
binned_wide = binned_df.pivot(
    index=["label", "bin", "low", "high", "true_mean"],
    columns="run_id",
    values=["MAE", "RMSE", "bias_pred_minus_true", "q90_abs_error"],
).reset_index()

In [ ]:
for metric in ["MAE", "RMSE", "q90_abs_error"]:
    binned_wide[(metric, "delta_M07_minus_M00")] = (
        binned_wide[(metric, "M07")]
        - binned_wide[(metric, "M00")]
    )

In [ ]:
binned_wide[("abs_bias", "M00")] = np.abs(
    binned_wide[("bias_pred_minus_true", "M00")]
)
binned_wide[("abs_bias", "M07")] = np.abs(
    binned_wide[("bias_pred_minus_true", "M07")]
)
binned_wide[("abs_bias", "delta_M07_minus_M00")] = (
    binned_wide[("abs_bias", "M07")]
    - binned_wide[("abs_bias", "M00")]
)

In [ ]:
for label in label_names:
    sub = binned_wide[binned_wide["label"] == label]

    plt.figure(figsize=(7.5, 4.5))
    plt.axhline(0, linewidth=1)
    plt.plot(
        sub["true_mean"],
        sub[("MAE", "delta_M07_minus_M00")],
        marker="o",
    )
    plt.xlabel(f"True {label}")
    plt.ylabel("Δ MAE: M07 − M00")
    plt.title(f"Per-bin MAE change — {label}")
    plt.tight_layout()

Interpretación inmediata:

* debajo de 0: M07 mejora;
* encima de 0: M07 empeora.

## 11*. Error quantiles

In [ ]:
test_quantiles = quantiles_df.query(
    "split == 'test' and space == 'physical'"
)

test_quantiles.sort_values(["label", "run_id"])

## 12*. Model cost

In [ ]:
def count_parameters_from_checkpoint(checkpoint_path):
    import torch

    ckpt = torch.load(checkpoint_path, map_location="cpu")
    state = ckpt["model_state_dict"]

    return sum(t.numel() for t in state.values())

In [ ]:
for run_id, run in RUNS.items():
    n_params = count_parameters_from_checkpoint(run["checkpoint"])
    print(f"{run_id}: {n_params:,} parameters")

## Conclusion

M07 ofrece una mejora real pero limitada:

* MSE global test: −1.67 %.
* MSE de chi_eff: −2.42 %.
* El bootstrap pareado sólo confirma una mejora clara para chi_eff; los intervalos de chirp_mass y total_mass cruzan cero.
* La mejora aparece también en validación y calibración, por lo que no parece un artefacto exclusivo del test.
* La pendiente de chi_eff apenas cambia: 0.7818 → 0.7838.
* La contracción robusta tampoco cambia materialmente:
std(pred)/std(true): 0.8855 → 0.8848;
rango 1–99 %: 0.8498 → 0.8518.
* En total_mass, la pendiente empeora: 0.9150 → 0.9036.

Por tanto:

Las cabezas no lineales independientes ayudan ligeramente a chi_eff, pero no resuelven la regresión hacia la media. El cuello de botella principal está antes de las heads, probablemente en la representación temporal producida por el encoder.